# '02.-Actualización Contable Marine' - PROGRAM 02

##  Overview
In this program, we use the bd_update.pkl processed in the previous program.
After that, load and prepare the Accounting information.
Cross the information of bd_update and the account information.
Finally, create the formulated variables and export.
This program creates the final database.

##  Execution Flow
1. Library Import.
2. Path Definition and Macrovariables. 
3. Data import (bd_update.pkl & Account Information). 
4. Data Format.
5. Data Cross.
6. Creation of formulated variables.
7. OSLR and Dates validation.
8. Final adjustments.
9. Final Export.

##  Output
- Excel file with the final processed database of the month. ({AñoMes}_Siniestros_Marine.xlsx)
- Pickle file with the final processed database of the month. ({AñoMes}_Siniestros_Marine.pkl)
- Excel file with relevant information for incidences analysis. (actuarial.xlsx)

## 1. Library import

In [350]:
## =============================================================================
# Section 1: Library import
# =============================================================================

# === Optional: Clean all the enviroment prior running === #
%reset -f                                                  
# ======================================================== #
import os
import pandas as pd
#import dtale
import numpy as np
import timeit
import sqlite3

start_time = timeit.default_timer() # Timer

## 2. Path Definition and Macrovariables 

In [351]:
## =============================================================================
# Section 2: Path Definition and Macrovariables. 
# =============================================================================
print('===============================================================================================')
print('Inicio del proceso')
start_time = timeit.default_timer() # Timer

# ================= Edit variables month================================ #
AñoMes = 202509
AñoMes_ant = 202504

# Convert the variables in datetime 
date_start_AñoMes = pd.to_datetime(str(AñoMes), format='%Y%m') + pd.offsets.MonthBegin(0)
date_start_AñoMes_ant = pd.to_datetime(str(AñoMes_ant), format='%Y%m') + pd.offsets.MonthBegin(0)
date_start_suma = date_start_AñoMes_ant + pd.offsets.MonthBegin(1) # This is an Aux Variable and it should be used when we are proccesing more than 1 month

print(f'Fecha de inicio para considerar en la suma: {date_start_suma}')

# =================Routes definition========================================= #

print('===============================================================================================')
# Define your project directory path as a variable
directorio_proyecto =  "C:/Users/IKAL14/Documents/Integral/Marine"   #C:/Users/KOT23/Documents/Integral

# Change the current working directory to the project directory
os.chdir(directorio_proyecto) 

# Verify that the current directory is the project directory
print(f"Directorio actual de trabajo: {os.getcwd()}")

# Processed files path
ruta_procesados = rf"{directorio_proyecto}/Procesados" 
print(f"Ruta de archivos procesados: {ruta_procesados}")

#Ruta de Bases
ruta_bases= rf"{directorio_proyecto}/Bases"
print(f"Ruta de bases: {ruta_bases}")

# Incidences files path
ruta_incidencias = rf"{directorio_proyecto}/Incidencias/{AñoMes}" 
print(f"Ruta de Incidencias: {ruta_incidencias}")

# Validation files path
route_validations = rf"{directorio_proyecto}/Validaciones/{AñoMes}" 
print(f"Ruta de Validaciones: {route_validations}")

# Catalog files path
route_catalog = rf'{directorio_proyecto}/Catalogos'
print(f'Ruta de catalogos: {route_catalog}')

# Actuarial database path
route_actuarial = rf'C:/Users/IKAL14/Documents/Integral/Insumos'
print(f'Ruta de base actuarial: {route_actuarial}')

# Payments information
route_account = fr'C:/Users/IKAL14/Documents/Integral/Insumos/Contabilidad/{AñoMes}'
print(f'Ruta de base de payments: {route_account}')

# 'Pruebas' Route
route_pruebas = rf"{directorio_proyecto}/Procesados/{AñoMes}/pruebas" 
# Verify if the path exists
if not os.path.exists(route_pruebas):
    os.makedirs(route_pruebas)
    print(f"Carpeta creada: {route_pruebas}")
else:
    print(f"La carpeta ya existe: {route_pruebas}")


# This DB includes all of the records that can be updated
df_update_db = pd.read_pickle(f'{ruta_procesados}/{AñoMes}/bd_update.pkl')
df_update_db['CLAIM NUMBER'] = df_update_db['CLAIM NUMBER'].astype(str)
df_update_db['LoB-Inward'] = df_update_db['LoB-Inward'].astype(str)

Inicio del proceso
Fecha de inicio para considerar en la suma: 2025-05-01 00:00:00
Directorio actual de trabajo: C:\Users\IKAL14\Documents\Integral\Marine
Ruta de archivos procesados: C:/Users/IKAL14/Documents/Integral/Marine/Procesados
Ruta de bases: C:/Users/IKAL14/Documents/Integral/Marine/Bases
Ruta de Incidencias: C:/Users/IKAL14/Documents/Integral/Marine/Incidencias/202509
Ruta de Validaciones: C:/Users/IKAL14/Documents/Integral/Marine/Validaciones/202509
Ruta de catalogos: C:/Users/IKAL14/Documents/Integral/Marine/Catalogos
Ruta de base actuarial: C:/Users/IKAL14/Documents/Integral/Insumos
Ruta de base de payments: C:/Users/IKAL14/Documents/Integral/Insumos/Contabilidad/202509
La carpeta ya existe: C:/Users/IKAL14/Documents/Integral/Marine/Procesados/202509/pruebas


## 3. Data import

In [352]:
# =============================================================================
# Section 3: Data import
# =============================================================================

# ======== Import of update database ======== #

# Normalizar CLAIM NUMBER antes de KEY LOB ==========
if 'CLAIM NUMBER' in df_update_db.columns:
    df_update_db['CLAIM NUMBER'] = (
        df_update_db['CLAIM NUMBER']
        .astype(str)
        .str.strip()
        .str.lstrip("'")  # Eliminar comilla inicial puesta por Excel
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

# Normalizar INWARD POLICY N° (quitar comilla de Excel) ==========
if 'INWARD POLICY N°' in df_update_db.columns:
    df_update_db['INWARD POLICY N°'] = (
        df_update_db['INWARD POLICY N°']
        .astype(str)
        .str.strip()
        .str.lstrip("'")  # Eliminar comilla inicial puesta por Excel
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

# Normalizar SUBSIDIARY (quitar comilla de Excel) ==========
if 'SUBSIDIARY' in df_update_db.columns:
    df_update_db['SUBSIDIARY'] = (
        df_update_db['SUBSIDIARY']
        .astype(str)
        .str.strip()
        .str.lstrip("'")  # Eliminar comilla inicial puesta por Excel
        .str.strip()
    )

# ── Diccionarios de normalización (sincronizados con Notebook 4) ─────────────
import re as _re

# cover_map: normaliza COVER a valores canónicos (igual a NB4 cell 9)
cover_map = {
    'CASCO Y MAQ.'               : 'CASCO Y MAQ.',
    'CASCO Y MAQUINARIA'         : 'CASCO Y MAQ.',
    'CASCO'                      : 'CASCO Y MAQ.',
    'GASTOS DE SALVAMENTO'       : 'CASCO Y MAQ.',
    'DAÑOS A LA MAQUINARIA'      : 'CASCO Y MAQ.',
    'P&I'                        : 'P&I',
    'PANDI'                      : 'P&I',
    'CARGO'                      : 'CARGO',
    'TRANSPORTE'                 : 'CARGO',
    'TRANSPORTE / CONTAMINACION' : 'CARGO',
    'TRANSPORTE/CONTAMINACION'   : 'CARGO',
    'CARGA POLIETILENO'          : 'CARGA POLIETILENO',
    'CARGA'                      : 'CARGA',
    'POLIETILENO'                : 'CARGA POLIETILENO',
    'DEEP WATERS'                : 'DEEP WATERS',
    'AGUAS PROFUNDAS'            : 'DEEP WATERS',
    'JACK-UPS'                   : 'JACK-UPS(DAÑO FISICO)',
    'JACK UPS'                   : 'JACK-UPS(DAÑO FISICO)',
    'PLATAFORMAS MOVILES'        : 'JACK-UPS(DAÑO FISICO)',
    'RC FLETADORES'              : 'RC FLETADORES(PEMEX)',
    'FLETADORES PMI'             : 'RC FLETADORES(PMI)',
    'FLETADORESPMI'              : 'RC FLETADORES(PMI)',
    'FLETADORES RC PMI'          : 'RC FLETADORES(PMI)',
    'EQUIPO FERROVIARIO'         : 'EQUIPO FERROVIARIO(DAÑO FÍSICO)',
    'FERREO'                     : 'EQUIPO FERROVIARIO(DAÑO FÍSICO)',
}

# maplobinward: COVER canónico → LoB-Inward final (igual a NB1 cell 24)
maplobinward = {
    'CASCO Y MAQ.'                    : 'CASCO Y MAQ.',
    'GASTOS DE SALVAMENTO'            : 'CASCO Y MAQ.',
    'EQUIPO FERROVIARIO(DAÑO FÍSICO)' : 'EQUIPO FERROVIARIO(DAÑO FÍSICO)',
    'DEEP WATERS'                     : 'DEEP WATERS',
    'CARGA'                           : 'CARGA',
    'CARGA POLIETILENO'               : 'CARGA',
    'JACK-UPS(DAÑO FISICO)'           : 'JACK-UPS(DAÑO FISICO)',
    'CARGO'                           : 'CARGO',
    'P&I'                             : 'P&I',
    'RC FLETADORES(PEMEX)'            : 'P&I',
    'RC FLETADORES(PMI)'              : 'P&I',
}

def _apply_cover_lob(df_in, cover_col='COVER', lob_col='LoB-Inward'):
    # Aplica la logica de COVER y LoB-Inward sincronizada entre NB4 y NB1.
    # 1. Limpia y normaliza COVER con cover_map (NB4 cell 9)
    # 2. Deriva LoB-Inward desde COVER con maplobinward (NB1 cell 24)
    df_in = df_in.copy()

    # Unificar COVERAGE → COVER si el BDX usa el nombre alternativo
    if 'COVERAGE' in df_in.columns and cover_col not in df_in.columns:
        df_in = df_in.rename(columns={'COVERAGE': cover_col})
    elif 'COVERAGE' in df_in.columns and cover_col in df_in.columns:
        df_in[cover_col] = df_in[cover_col].fillna(df_in['COVERAGE'])

    if cover_col in df_in.columns:
        df_in[cover_col] = (
        df_in[cover_col]
        .fillna('NO ESPECIFICADO')
        .astype(str)
        .str.replace('\u200b', ' ', regex=False)
        .str.replace('\ufeff', ' ', regex=False)
        .str.replace('\xa0', ' ', regex=False)
        .str.replace(r'[\t\r]', ' ', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
        .str.upper()
        .replace(cover_map)
    )

    if lob_col in df_in.columns and cover_col in df_in.columns:
        df_in[lob_col] = df_in[lob_col].astype(str).str.strip().str.upper()
        mask_vacio = (
            df_in[lob_col].isin(['', 'NAN', 'NONE', 'NO ESPECIFICADO'])
            | df_in[lob_col].isna()
        )
        df_in.loc[mask_vacio, lob_col] = df_in.loc[mask_vacio, cover_col]
        df_in[lob_col] = (
            df_in[lob_col]
            .map(maplobinward)
            .fillna(df_in[lob_col])
        )

    return df_in

# Normalizar COVER y LoB-Inward para todos los registros del BDX
df_update_db = _apply_cover_lob(df_update_db)
n_lob_vacios = (
    df_update_db['LoB-Inward'].isna()
    | df_update_db['LoB-Inward'].astype(str).str.strip().isin(['', 'nan', 'NAN'])
).sum()
print(f'LoB-Inward vacios tras normalización: {n_lob_vacios}')

# Crear KEY LOB con CLAIM NUMBER y LoB-Inward ya normalizados
df_update_db['KEY LOB'] = df_update_db['CLAIM NUMBER'] + '-' + df_update_db['LoB-Inward']


# Deduplicar filas con el mismo KEY LOB: combinar tomando primer valor no-nulo de cada columna
dups_mask = df_update_db.duplicated(subset='KEY LOB', keep=False)
n_dup_keys = df_update_db.loc[dups_mask, 'KEY LOB'].nunique()
if n_dup_keys > 0:
    print(f'\n  KEY LOBs duplicados en BDX (antes de combinar): {n_dup_keys}')
    for k in df_update_db.loc[dups_mask, 'KEY LOB'].unique():
        print(f'   - {k}')

    def first_non_null(series):
        # Devuelve el primer valor no-nulo/vacio de la serie
        for val in series:
            try:
                if pd.notna(val) and str(val).strip() not in ('', 'nan', 'NaT', 'None'):
                    return val
            except Exception:
                if val is not None:
                    return val
        return series.iloc[0]

    registros_antes = len(df_update_db)
    df_update_db = (
        df_update_db
        .groupby('KEY LOB', sort=False)
        .agg(first_non_null)
        .reset_index()  # trae KEY LOB de vuelta como columna
    )
    print(f'  Combinacion completada: {registros_antes} registros -> {len(df_update_db)} registros')

    # Normalizar CLAIM NUMBER
    df_update_db['CLAIM NUMBER'] = (
        df_update_db['CLAIM NUMBER'].astype(str).str.strip().str.lstrip("'")
        .str.replace(r'\s+', '', regex=True)
    )

    # Reconstruir KEY LOB con los valores ya normalizados
    df_update_db['KEY LOB'] = df_update_db['CLAIM NUMBER'] + '-' + df_update_db['LoB-Inward']
    print('  Normalizaciones re-aplicadas (COVER, LoB-Inward, CLAIM NUMBER, KEY LOB)')
else:
    print('  No hay KEY LOBs duplicados en BDX')


# Rename of the columns
df_update_db.rename(columns= {
                              'DEDUCTIBLE': f'DEDUCTIBLE {AñoMes_ant}' , 'DEDUCTIBLE BDX': f'DEDUCTIBLE {AñoMes}',
                              'GROSS RESERVE': f'GROSS RESERVE {AñoMes_ant}', 'GROSS RESERVE BDX': f'GROSS RESERVE {AñoMes}',
                              'Cumulative EXPENSES PAID':f'Cumulative EXPENSES PAID {AñoMes_ant}',
                              'Cumulative VALUATION EXPENSES':f'Cumulative VALUATION EXPENSES {AñoMes_ant}',
                              'Cumulative CLAIMS PAID':f'Cumulative CLAIMS PAID {AñoMes_ant}',
                              'OSLR Inward':f'OSLR Inward {AñoMes_ant}'}, inplace= True)

# ======== Import of account information ======== #
# Initialize a dictionary with the sheets of information
dict_sheets = pd.read_excel(fr'{route_account}/Payments_{AñoMes}.xlsx', sheet_name= None)
# Split the dictionary in dataframes
df_AE = dict_sheets['AE'] 
df_CL = dict_sheets['CL']
df_VA = dict_sheets['VA']

# Cleaning
del dict_sheets


LoB-Inward vacios tras normalización: 0

  KEY LOBs duplicados en BDX (antes de combinar): 71
   - 10205839-CARGO
   - 63967756-CARGO
   - 63968341-PANDI
   - 351003041562-CARGO
   - 351003041999-CARGO
   - 3417/2019-CARGA
   - 621/2021-CARGA
   - 4485/2023-P&I
   - 4308/2023-P&I
   - 2452/2024-P&I
   - 101815/2025-CARGA
   - 101847/2025-CARGA
   - 101919/2025-CARGA
   - 102048/2025-CARGA
   - 102152/2025-CARGA
   - 102155/2025-CARGA
   - 102189/2025-CARGA
   - 102191/2025-CARGA
   - 102260/2025-CARGA
   - 102262/2025-CARGA
   - 102304/2025-CARGA
   - 102375/2025-CARGA
   - 102450/2025-CARGA
   - 102508/2025-CARGA
   - 102558/2025-CARGA
   - 102660/2025-CARGA
   - 102663/2025-CARGA
   - 102692/2025-CARGA
   - 102693/2025-CARGA
   - 102740/2025-CARGA
   - 102741/2025-CARGA
   - 102748/2025-CARGA
   - 102750/2025-CARGA
   - 102961/2025-CARGA
   - 103031/2025-CARGA
   - 103063/2025-CARGA
   - 103064/2025-CARGA
   - 103065/2025-CARGA
   - 103116/2025-CARGA
   - 103172/2025-CARGA
   - 10331

## 4. Data format

In [353]:
# =============================================================================
# Section 4: Data Format
# =============================================================================

# =================== Account information =================== #

# Agrupamos los DataFrames en una lista para aplicar limpieza masiva
# Al ser objetos mutables, modificar 'df' dentro de un bucle actualizará el DataFrame original.
list_df_payments = [df_AE, df_CL, df_VA]

# ======= Format Variables ======= #

# === Date Variables === #
cols_date = ['Cover period from', 'Cover period to','Claim Date','Claim Report', 'Date of payment','Payment request date']

for df in list_df_payments:
    for col in cols_date:
        if col in df.columns:
            # Strip a los strings (ignorando nulos) y conversión a datetime
            df[col] = pd.to_datetime(
                df[col].astype(str).str.strip().replace({'nan': None, 'NaT': None}), 
                dayfirst=True, 
                errors='coerce'
            )

# Validar que todas las columnas de fecha existan
for i, df in enumerate(list_df_payments):
    missing = [col for col in cols_date if col not in df.columns]
    if missing:
        print(f"❌ DataFrame {i} no tiene las columnas: {missing}")
    else:
        print(f"✅ DataFrame {i} tiene todas las columnas de fecha")

# === String Variables === #
cols_string = ['Claims Reference', 'Lob-Inward']

for df in list_df_payments:
    for col in cols_string:
        df[col] = df[col].apply(
            lambda x: str(x).strip().lstrip("'").replace(' ', '') if pd.notna(x) and str(x).strip().lower() != 'nan' else x  # lstrip quita comilla de Excel
        )
    
    # Clean 'Policy N°' to avoid mismatches in subsidiary filter
    df['Policy N°'] = df['Policy N°'].astype(str).str.strip().str.lstrip("'")  # lstrip quita comilla de Excel


# ======= Data Cleaning ======= #

# ======= Data Cleaning ======= #
# Añade las dos columnas faltantes aquí:
cols_payment_base = [
    'Policy N°',
    'Claims Reference',
    'Claim Date',
    'Claim Report',
    'Description',
    'Lob-Inward',
    'Other 1 Cover',
    'Currency',
    'Payment request date',
    'Date of payment',
    'Amount USD (CORRECT)',
    'Status',
    'Cover period from',  # <--- Añadido
    'Cover period to'     # <--- Añadido
]

# ========== Mapa canónico de normalización de Lob-Inward ==========
lob_normalize_map = {
    'cascoymaquinaria':   'CASCO Y MAQUINARIA',
    'cascoy maquinaria':  'CASCO Y MAQUINARIA',
    'casco y maquinaria': 'CASCO Y MAQUINARIA',
    'casco':              'CASCO Y MAQUINARIA',
    'p&i':                'PANDI',
    'pandi':              'PANDI',
    'dw':                 'DEEP WATERS',
    'deep waters':        'DEEP WATERS',
    'deepwaters':         'DEEP WATERS',
    'cargo':              'CARGO',
    'plataformas':        'PLATAFORMAS',
    'floteles':           'FLOTELES',
}

for df in list_df_payments:
    normalized = (
        df['Lob-Inward']
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r'\s+', ' ', regex=True)
        .map(lob_normalize_map)
    )
    mask_mapped = normalized.notna()
    df.loc[mask_mapped, 'Lob-Inward'] = normalized[mask_mapped]

# Audit: print unique LoB values post-normalization
print("\n=== Valores unicos de Lob-Inward post-normalizacion ===")
for name, df in zip(['df_AE', 'df_CL', 'df_VA'], list_df_payments):
    print(f"\n{name}:")
    print(df['Lob-Inward'].value_counts(dropna=False).to_string())

# === Create KEY LOB: Claims Reference + Lob-Inward ===
for df in list_df_payments:
    df['KEY LOB'] = df['Claims Reference'].astype(str) + "-" + df['Lob-Inward'].astype(str)

# Verify KEY LOB was created successfully
print("\n✓ Verificando que KEY LOB existe en todos los dataframes:")
for name, df in zip(['df_AE', 'df_CL', 'df_VA'], list_df_payments):
    estado = "contiene" if 'KEY LOB' in df.columns else "NO contiene"
    print(f"  {'✓' if estado == 'contiene' else '✗'} {name} {estado} KEY LOB")

# Validate: detect "nan" contamination in KEY LOB
print("\n=== Validacion: KEY LOBs con nan (contaminacion por NaN) ===")
for name, df in zip(['df_AE', 'df_CL', 'df_VA'], list_df_payments):
    bad = df[df['KEY LOB'].str.contains('nan', case=False, na=False)]
    if not bad.empty:
        print(f"  ⚠️ {name}: {len(bad)} registros con nan en KEY LOB — revisar Claims Reference vacio")
        print(bad[['Claims Reference', 'Lob-Inward', 'KEY LOB', 'Amount USD (CORRECT)']].head())
    else:
        print(f"  ✅ {name}: sin contaminacion de nan")

# === Filtering by date ===
cols_payment = cols_payment_base + ['KEY LOB']

# Aplicamos la máscara
mask_AE = (df_AE['Payment request date'] >= date_start_suma) | pd.isna(df_AE['Payment request date'])
# Intersectamos las columnas deseadas con las que realmente existen en df_AE
cols_existentes_AE = df_AE.columns.intersection(cols_payment)
df_AE_filter = df_AE.loc[mask_AE, cols_existentes_AE].copy()

mask_CL = (df_CL['Payment request date'] >= date_start_suma) | pd.isna(df_CL['Payment request date'])
cols_existentes_CL = df_CL.columns.intersection(cols_payment)
df_CL_filter = df_CL.loc[mask_CL, cols_existentes_CL].copy()

mask_VA = (df_VA['Payment request date'] >= date_start_suma) | pd.isna(df_VA['Payment request date'])
cols_existentes_VA = df_VA.columns.intersection(cols_payment)
df_VA_filter = df_VA.loc[mask_VA, cols_existentes_VA].copy()

# NaT audit report
print("\n=== NaT en Payment request date (pasan el filtro por isna) ===")
for name, df in [('AE', df_AE), ('CL', df_CL), ('VA', df_VA)]:
    nat_count = df['Payment request date'].isna().sum()
    print(f"  {name}: {nat_count} NaT de {len(df)} registros ({nat_count/max(len(df),1):.1%})")

# === Filter by Lob-Inward Marine ===
list_lob_marine = ['CASCO Y MAQUINARIA', 'CARGO', 'PANDI', 'DEEP WATERS', 'PLATAFORMAS', 'FLOTELES']
list_lob_nomarine = set()

for df in [df_AE_filter, df_CL_filter, df_VA_filter]:
    valores_fuera = df.loc[~df['Lob-Inward'].isin(list_lob_marine), 'Lob-Inward'].unique()
    list_lob_nomarine.update(valores_fuera)

print(f'\nLoB consideradas para Marine: {list_lob_marine}')
print(f'LoB NO consideradas para Marine: {list_lob_nomarine}')

# Apply marine filter
df_AE_filter = df_AE_filter[df_AE_filter['Lob-Inward'].isin(list_lob_marine)]
df_CL_filter = df_CL_filter[df_CL_filter['Lob-Inward'].isin(list_lob_marine)]
df_VA_filter = df_VA_filter[df_VA_filter['Lob-Inward'].isin(list_lob_marine)]

# === Filter to exclude subsidiary company policies ===
list_subsidiary = ['25300 30028443', '25300 30031974']
subsidiary_clean = [s.strip() for s in list_subsidiary]

df_AE_filter = df_AE_filter[~df_AE_filter['Policy N°'].str.strip().isin(subsidiary_clean)]
df_CL_filter = df_CL_filter[~df_CL_filter['Policy N°'].str.strip().isin(subsidiary_clean)]
df_VA_filter = df_VA_filter[~df_VA_filter['Policy N°'].str.strip().isin(subsidiary_clean)]

# === Group by KEY LOB ===
df_AE_final = df_AE_filter.groupby(['KEY LOB', 'Policy N°'])['Amount USD (CORRECT)'].sum().reset_index()
df_AE_final.rename(columns={'Amount USD (CORRECT)': f'Cumulative EXPENSES PAID {AñoMes}'}, inplace=True)

df_CL_final = df_CL_filter.groupby(['KEY LOB', 'Policy N°'])['Amount USD (CORRECT)'].sum().reset_index()
df_CL_final.rename(columns={'Amount USD (CORRECT)': f'Cumulative CLAIMS PAID {AñoMes}'}, inplace=True)

df_VA_final = df_VA_filter.groupby(['KEY LOB', 'Policy N°'])['Amount USD (CORRECT)'].sum().reset_index()
df_VA_final.rename(columns={'Amount USD (CORRECT)': f'Cumulative VALUATION EXPENSES {AñoMes}'}, inplace=True)

# === Final merge via concat + groupby ===
df_all = pd.concat([df_AE_final, df_CL_final, df_VA_final], ignore_index=True)
df_conta_final = (
    df_all
    .groupby('KEY LOB', as_index=False)
    .agg({
        f'Cumulative EXPENSES PAID {AñoMes}':      'sum',
        f'Cumulative CLAIMS PAID {AñoMes}':        'sum',
        f'Cumulative VALUATION EXPENSES {AñoMes}': 'sum',
        'Policy N°': 'first'
    })
)

# === Reconstruct LoB-Inward on df_conta_final ===
df_conta_final['CLAIM NUMBER'] = df_conta_final['KEY LOB'].str.split('-', n=1).str[0]

df_update_db.columns = df_update_db.columns.str.strip()
df_update_db = df_update_db.drop(columns=['CLAIM NUMBER_y', 'LoB-Inward_y'], errors='ignore')
df_update_db = df_update_db.rename(columns={'CLAIM NUMBER_x': 'CLAIM NUMBER', 'LoB-Inward_x': 'LoB-Inward'})

df_lob_update = (
    df_update_db
    .drop_duplicates(subset='CLAIM NUMBER')
    .set_index('CLAIM NUMBER')['LoB-Inward']
)

df_conta_final['LoB-Inward'] = df_conta_final['CLAIM NUMBER'].map(df_lob_update)

df_lob_all = (
    df_all
    .drop_duplicates(subset='KEY LOB')
    .assign(LoB_Inward=lambda x: x['KEY LOB'].str.split('-', n=1).str[1])
    .set_index('KEY LOB')['LoB_Inward']
)

mask = df_conta_final['LoB-Inward'].isna()
df_conta_final.loc[mask, 'LoB-Inward'] = df_conta_final.loc[mask, 'KEY LOB'].map(df_lob_all)

df_conta_final['KEY LOB'] = (
    df_conta_final['CLAIM NUMBER'].astype(str)
    + "-"
    + df_conta_final['LoB-Inward'].astype(str)
)

# === Final format ===
cols_to_fill = [f'Cumulative EXPENSES PAID {AñoMes}', f'Cumulative CLAIMS PAID {AñoMes}', f'Cumulative VALUATION EXPENSES {AñoMes}']
df_conta_final[cols_to_fill] = df_conta_final[cols_to_fill].fillna(0)

df_conta_final['FLAG CONTA'] = 1

# Print Validation
print(f"\nTotal ${round(df_conta_final[f'Cumulative EXPENSES PAID {AñoMes}'].sum(), 2)} en Cumulative EXPENSES PAID")
print(f"Total ${round(df_conta_final[f'Cumulative CLAIMS PAID {AñoMes}'].sum(), 2)} en Cumulative CLAIMS PAID")
print(f"Total ${round(df_conta_final[f'Cumulative VALUATION EXPENSES {AñoMes}'].sum(), 2)} en Cumulative VALUATION EXPENSES")

# ===== Handmade Adjustments ===== #
if AñoMes == 202503:
    replacements = {
        '323372640000025-CARGO': '323372640000025-CARGA POLIETILENO',
        '323372640000051-CARGO': '323372640000051-CARGA POLIETILENO'
    }
    df_conta_final['KEY LOB'] = df_conta_final['KEY LOB'].replace(replacements)

# === Metadata lookup para registros nuevos desde contabilidad ===
df_conta_meta = (
    pd.concat([df_AE_filter, df_CL_filter, df_VA_filter], ignore_index=True)
    .groupby('KEY LOB', as_index=False)
    .agg({
        'Policy N°':         'first',
        'Claims Reference': 'first',
        'Description': 'first',
        'Other 1 Cover': 'first',
        'Claim Date':        'first',
        'Claim Report':      'first',
        'Cover period from': 'first',
        'Cover period to':   'first',
    })
    .rename(columns={
        'Policy N°':         'INWARD POLICY N°',
        'Claims Reference': 'CLAIM NUMBER',
        'Description': 'DESCRIPTION',
        'Other 1 Cover': 'COVER',
        'Claim Date':        'DATE OF LOSS',
        'Claim Report':      'DATE OF REPORT',
        'Cover period from': 'POLICY PERIOD START DATE',
        'Cover period to':   'POLICY PERIOD END DATE',
    })
)
print(f"\ndf_conta_meta creado: {len(df_conta_meta)} registros con metadatos de contabilidad")

df_conta_final = df_conta_final.drop(columns=["CLAIM NUMBER", "LoB-Inward", "Policy N°"], errors='ignore')


✅ DataFrame 0 tiene todas las columnas de fecha
✅ DataFrame 1 tiene todas las columnas de fecha
✅ DataFrame 2 tiene todas las columnas de fecha

=== Valores unicos de Lob-Inward post-normalizacion ===

df_AE:
Lob-Inward
ONL                   63
ONPD                  28
NaN                   10
OFFPD                  7
PANDI                  4
CARGO                  3
CASCO Y MAQUINARIA     2
ONLIT                  2
DEEP WATERS            2
OFFL                   1
OEE                    1
PLATAFORMAS            1

df_CL:
Lob-Inward
CARGO                  81
ONPD                   20
EE-T                   20
ONL                    16
NaN                    12
RCTerceros             11
EE                      8
OFFPD                   7
CASCO Y MAQUINARIA      3
PANDI                   3
Remocióndeescombros     2
GastosdeMitigación      1
ONLIT                   1
OEE                     1
Gastosextras            1

df_VA:
Lob-Inward
NaN                   11
ONL                    3
ON

In [354]:
# --- SOLUCIÓN AUTOMÁTICA PARA LA COLUMNA 'FLAG CONTA' ---
# Esto asegura que siempre exista la columna correcta, sin importar el estado previo del merge
col_base = 'FLAG CONTA'
col_old = f'{col_base}_old'
col_new = f'{col_base}_new'

if col_old in df_update_db.columns and col_new in df_update_db.columns:
    df_update_db[col_base] = df_update_db[col_new].fillna(df_update_db[col_old])
    df_update_db.drop([col_old, col_new], axis=1, inplace=True)
elif col_new in df_update_db.columns:
    df_update_db.rename(columns={col_new: col_base}, inplace=True)
elif col_old in df_update_db.columns:
    df_update_db.rename(columns={col_old: col_base}, inplace=True)
# Si ya existe sin sufijos, no se hace nada

print('Consolidación de FLAG CONTA completada. Columnas actuales:')
print([col for col in df_update_db.columns if 'FLAG CONTA' in col])

# Consolidar FLAG CONTA correctamente
if 'FLAG CONTA_new' in df_update_db.columns and 'FLAG CONTA_old' in df_update_db.columns:
    df_update_db['FLAG CONTA'] = df_update_db['FLAG CONTA_new'].combine_first(df_update_db['FLAG CONTA_old'])
    df_update_db.drop(['FLAG CONTA_new', 'FLAG CONTA_old'], axis=1, inplace=True)
elif 'FLAG CONTA_new' in df_update_db.columns:
    df_update_db.rename(columns={'FLAG CONTA_new': 'FLAG CONTA'}, inplace=True)
elif 'FLAG CONTA_old' in df_update_db.columns:
    df_update_db.rename(columns={'FLAG CONTA_old': 'FLAG CONTA'}, inplace=True)

Consolidación de FLAG CONTA completada. Columnas actuales:
[]


## 5. Data Cross

In [355]:
# =============================================================================
# DIAGNÓSTICO: KEY LOBs que NO hacen match entre BDX y Contabilidad
# =============================================================================
print("=" * 80)
print("DIAGNÓSTICO DE MATCH ENTRE BDX Y CONTABILIDAD")
print("=" * 80)

keys_bdx   = set(df_update_db['KEY LOB'].dropna().unique())
keys_conta = set(df_conta_final['KEY LOB'].dropna().unique())

solo_en_conta = keys_conta - keys_bdx   # Registros que van a crear filas nuevas (duplicados)
solo_en_bdx   = keys_bdx - keys_conta   # Registros sin movimiento contable
en_ambos      = keys_bdx & keys_conta   # Match perfecto

print(f"\nTotal KEY LOB en BDX:          {len(keys_bdx)}")
print(f"Total KEY LOB en Contabilidad:  {len(keys_conta)}")
print(f"Con match perfecto:             {len(en_ambos)}")
print(f"SOLO en Contabilidad (= crean fila doble): {len(solo_en_conta)}")
print(f"SOLO en BDX (sin movimiento):   {len(solo_en_bdx)}")

if solo_en_conta:
    print("\n--- KEY LOBs de Contabilidad SIN MATCH en BDX ---")
    for k in sorted(solo_en_conta):
        claim_part = k.split('-')[0] if '-' in k else k
        lob_part   = k.split('-', 1)[1] if '-' in k else ''
        # Buscar si el claim existe en BDX con diferente LoB
        bdx_matches = [b for b in keys_bdx if b.startswith(claim_part + '-')]
        print(f"  Conta KEY LOB : {repr(k)}")
        if bdx_matches:
            print(f"  BDX (mismo claim, diferente LoB): {[repr(b) for b in bdx_matches]}")
        else:
            print(f"  BDX: sin registro con CLAIM NUMBER {repr(claim_part)}")
        print()


DIAGNÓSTICO DE MATCH ENTRE BDX Y CONTABILIDAD

Total KEY LOB en BDX:          4255
Total KEY LOB en Contabilidad:  46
Con match perfecto:             44
SOLO en Contabilidad (= crean fila doble): 2
SOLO en BDX (sin movimiento):   4211

--- KEY LOBs de Contabilidad SIN MATCH en BDX ---
  Conta KEY LOB : '324392640000098-CARGO'
  BDX: sin registro con CLAIM NUMBER '324392640000098'

  Conta KEY LOB : '6490/2022-CARGO'
  BDX: sin registro con CLAIM NUMBER '6490/2022'



In [356]:
# =============================================================================
# Section 5: Data Cross - VERSIÓN CORREGIDA Y DEPURADA
# =============================================================================

print('=' * 95)
print('Comenzamos con el cruce entre la base actualizable del mes y la información contable del mes')
print(f'Tenemos {len(df_update_db)} registros en la base actualizable del mes')
print(f'Tenemos {len(df_conta_final)} registros en la base contable del mes')

# =============================================================================
# DIAGNÓSTICO: Verificar columnas ANTES de cualquier operación
# =============================================================================
print('\n🔍 COLUMNAS EN df_update_db ANTES DE LIMPIEZA:')
print([col for col in df_update_db.columns if 'FLAG' in col or 'CONTA' in col])

# =============================================================================
# Normalización de columnas clave para evitar problemas
# =============================================================================
# Normalizar COLUMNAS (quitar comilla de Excel) ==========
normcolumn={'SUUBSIDIARY','STATUS','REF. PEMEX','LOCATION','DESCRIPTION'}
for col in normcolumn:
     if col in df_update_db.columns:
        df_update_db[col] = (
            df_update_db[col]
            .astype(str)
            .str.strip()
            .str.lstrip("'")  # Eliminar comilla inicial puesta por Excel
            .str.strip()
        )
        
# =============================================================================
# PASO 1: LIMPIEZA DE COLUMNAS DUPLICADAS (_y)
# =============================================================================
print('\n🧹 LIMPIEZA DE COLUMNAS _y:')
cols_to_drop = [col for col in df_update_db.columns if col.endswith('_y')]
if cols_to_drop:
    print(f"Eliminando: {cols_to_drop}")
    df_update_db = df_update_db.drop(columns=cols_to_drop, errors='ignore')
else:
    print("No hay columnas _y para eliminar")

# =============================================================================
# PASO 2: RENOMBRAR COLUMNAS _x (PROBLEMA IDENTIFICADO AQUÍ)
# =============================================================================
print('\n🔧 RENOMBRANDO COLUMNAS _x:')

#  Verificar qué columnas _x existen REALMENTE
cols_x_existentes = [col for col in df_update_db.columns if col.endswith('_x')]
print(f"Columnas _x encontradas: {cols_x_existentes}")

# Crear diccionario de rename solo para columnas que EXISTEN
cols_to_rename = {}
for col in cols_x_existentes:
    nueva_col = col.removesuffix('_x')  # FIX #4: removesuffix en lugar de rstrip para evitar corrupción de nombres
    cols_to_rename[col] = nueva_col
    print(f"  {col} → {nueva_col}")

if cols_to_rename:
    df_update_db = df_update_db.rename(columns=cols_to_rename)
    print(f"✓ Renombradas {len(cols_to_rename)} columnas")
else:
    print("No hay columnas _x para renombrar")

# =============================================================================
# VERIFICACIÓN: ¿FLAG CONTA existe ahora en df_update_db?
# =============================================================================
print('\n🔍 VERIFICACIÓN POST-LIMPIEZA:')
flag_conta_existe = 'FLAG CONTA' in df_update_db.columns
print(f"¿FLAG CONTA existe en df_update_db? {flag_conta_existe}")
if flag_conta_existe:
    print(f"  Valores únicos: {df_update_db['FLAG CONTA'].unique()}")
    print(f"  Nulos: {df_update_db['FLAG CONTA'].isna().sum()}")

# =============================================================================
# PASO 3: PREPARAR MERGE
# =============================================================================
print('\n📊 PREPARACIÓN DEL MERGE:')

cols_extra = [
    'KEY LOB',
    'FLAG CONTA',
    f'Cumulative EXPENSES PAID {AñoMes}',
    f'Cumulative CLAIMS PAID {AñoMes}',
    f'Cumulative VALUATION EXPENSES {AñoMes}'
]

# Verificar columnas en df_conta_final
cols_extra_ok = [c for c in cols_extra if c in df_conta_final.columns]
cols_missing = set(cols_extra) - set(cols_extra_ok)

print(f"✅ Columnas disponibles en df_conta_final: {cols_extra_ok}")
if cols_missing:
    print(f"❌ Columnas faltantes: {cols_missing}")

# =============================================================================
# PASO 4: REALIZAR MERGE
# =============================================================================
print('\n🔗 REALIZANDO MERGE:')

# Guardar información PRE-merge para comparación
registros_antes = len(df_update_db)
tenia_flag_conta = 'FLAG CONTA' in df_update_db.columns

print(f"Registros antes del merge: {registros_antes}")
print(f"FLAG CONTA existía antes del merge: {tenia_flag_conta}")

# MERGE con sufijos explícitos para control total
print(f"\n📊 INFO PRE-MERGE:")
print(f"  Registros en df_update_db: {len(df_update_db)}")
print(f"  Registros en df_conta_final: {len(df_conta_final)}")
print(f"  KEY LOB en df_conta_final: {df_conta_final['KEY LOB'].unique().tolist()}")

df_update_db = df_update_db.merge(
    df_conta_final[cols_extra_ok],
    on='KEY LOB',
    how='outer',  # ← CAMBIO CRUCIAL: outer para incluir registros nuevos
    suffixes=('_old', '_new')  # Sufijos más claros
)

# Asegurar consolidación inmediata de FLAG CONTA
if 'FLAG CONTA_old' in df_update_db.columns and 'FLAG CONTA_new' in df_update_db.columns:
    df_update_db['FLAG CONTA'] = df_update_db['FLAG CONTA_new'].fillna(
        df_update_db['FLAG CONTA_old']
    )
    df_update_db.drop(columns=['FLAG CONTA_old', 'FLAG CONTA_new'], inplace=True)

elif 'FLAG CONTA_new' in df_update_db.columns:
    df_update_db.rename(columns={'FLAG CONTA_new': 'FLAG CONTA'}, inplace=True)

elif 'FLAG CONTA_old' in df_update_db.columns:
    df_update_db.rename(columns={'FLAG CONTA_old': 'FLAG CONTA'}, inplace=True)

#Columnas duplicadas detectadas
if not df_update_db.columns.is_unique:
    print("⚠️ Columnas duplicadas detectadas. Corrigiendo...")
    df_update_db = df_update_db.loc[:, ~df_update_db.columns.duplicated()]

registros_despues = len(df_update_db)
print(f"Registros después del merge: {registros_despues}")
print(f"✓ Incremento de registros: {registros_despues - registros_antes}")

# Diagnóstico: mostrar registros con movimientos
nuevos_en_merge = df_update_db[df_update_db['FLAG CONTA'] == 1].copy()
print(f"\n📲 Registros con movimientos financieros (FLAG CONTA=1): {len(nuevos_en_merge)}")
if len(nuevos_en_merge) > 0:
    print("  Detalles de KEY LOB con movimientos:")
    for key in nuevos_en_merge['KEY LOB'].unique():
        mov_exp = nuevos_en_merge[nuevos_en_merge['KEY LOB']==key][f'Cumulative EXPENSES PAID {AñoMes}'].sum()
        mov_claim = nuevos_en_merge[nuevos_en_merge['KEY LOB']==key][f'Cumulative CLAIMS PAID {AñoMes}'].sum()
        mov_val = nuevos_en_merge[nuevos_en_merge['KEY LOB']==key][f'Cumulative VALUATION EXPENSES {AñoMes}'].sum()
        total_mov = mov_exp + mov_claim + mov_val
        print(f"    {key}: ${total_mov:,.2f} (EXPENSES={mov_exp}, CLAIMS={mov_claim}, VAL={mov_val})")

# =============================================================================
# PASO 5: MANEJO DE COLUMNAS DUPLICADAS POST-MERGE
# =============================================================================
print('\n🔧 CONSOLIDANDO COLUMNAS DUPLICADAS:')

for col_base in ['FLAG CONTA', 
                 f'Cumulative EXPENSES PAID {AñoMes}',
                 f'Cumulative CLAIMS PAID {AñoMes}',
                 f'Cumulative VALUATION EXPENSES {AñoMes}']:
    
    col_old = f'{col_base}_old'
    col_new = f'{col_base}_new'
    
    # Caso 1: Existen ambas versiones (old y new)
    if col_old in df_update_db.columns and col_new in df_update_db.columns:
        print(f"\n  Procesando {col_base}:")
        print(f"    - {col_old} existe: Sí")
        print(f"    - {col_new} existe: Sí")
        
        # Priorizar _new (viene del merge actual)
        # Si _new es null, usar _old
        df_update_db[col_base] = df_update_db[col_new].fillna(df_update_db[col_old])
        
        # Eliminar versiones old y new
        df_update_db = df_update_db.drop(columns=[col_old, col_new])
        print(f"    ✓ Consolidado en {col_base}")
    
    # Caso 2: Solo existe _new (merge exitoso, no había columna previa)
    elif col_new in df_update_db.columns:
        print(f"\n  {col_base}: Solo existe _new, renombrando")
        df_update_db = df_update_db.rename(columns={col_new: col_base})
    
    # Caso 3: Solo existe _old (no hubo merge, o columna no estaba en df_conta_final)
    elif col_old in df_update_db.columns:
        print(f"\n  {col_base}: Solo existe _old, renombrando")
        df_update_db = df_update_db.rename(columns={col_old: col_base})
    
    # Caso 4: Ya existe sin sufijos (ideal)
    elif col_base in df_update_db.columns:
        print(f"\n  {col_base}: Ya existe sin sufijos ✓")
    
    # Caso 5: No existe en ninguna forma
    else:
        print(f"\n  ⚠️  {col_base}: NO EXISTE en ninguna forma")

# =============================================================================
# PASO 6: RELLENAR NULOS CON 0
# =============================================================================
print('\n📝 RELLENANDO VALORES NULOS:')

cols_to_fill = [
    'FLAG CONTA',
    f'Cumulative EXPENSES PAID {AñoMes}',
    f'Cumulative CLAIMS PAID {AñoMes}',
    f'Cumulative VALUATION EXPENSES {AñoMes}'
]

for col in cols_to_fill:
    if col in df_update_db.columns:
        nulos_antes = df_update_db[col].isna().sum()
        df_update_db[col] = df_update_db[col].fillna(0)
        nulos_despues = df_update_db[col].isna().sum()
        print(f"  ✓ {col}: {nulos_antes} → {nulos_despues} nulos")
    else:
        print(f"  ⚠️  {col}: NO EXISTE - no se puede rellenar")

# =============================================================================
# PASO 6.5: RELLENAR REGISTROS NUEVOS DESDE CONTABILIDAD
# =============================================================================
print('\n🆕 RELLENANDO REGISTROS NUEVOS (solo de contabilidad):')

# Identificar registros que vienen SOLO de contabilidad (KEY LOB no estaba en bd_update original)
# Esto sucede cuando otros campos están vacíos
new_claims_mask = (
    (df_update_db['FLAG CONTA'] == 1) &  # Tiene FLAG CONTA (vino de contabilidad)
    (df_update_db[[f'GROSS RESERVE {AñoMes}', f'DEDUCTIBLE {AñoMes}']].isna().any(axis=1))  # Pero faltan datos de reserva
)

new_claims_count = new_claims_mask.sum()
print(f"  Registros nuevos identificados: {new_claims_count}")

if new_claims_count > 0:
    # Mostrar los siniestros nuevos
    new_claims_list = df_update_db.loc[new_claims_mask, 'KEY LOB'].unique()
    print(f"  Siniestros nuevos:")
    for claim in new_claims_list:
        print(f"    - {claim}")

    # --- Rellenar con metadatos reales de contabilidad ---
    # Deduplicar df_conta_meta por KEY LOB para evitar que .map() devuelva Series
    meta_indexed = df_conta_meta.drop_duplicates(subset='KEY LOB').set_index('KEY LOB')
    meta_key_series = df_update_db.loc[new_claims_mask, 'KEY LOB']

    # CLAIM NUMBER: extraer de KEY LOB para registros nuevos sin claim number
    mask_cn_vacio = new_claims_mask & (
        df_update_db['CLAIM NUMBER'].isna() |
        df_update_db['CLAIM NUMBER'].astype(str).str.strip().isin(['', 'nan', 'None'])
    )
    if mask_cn_vacio.any():
        df_update_db.loc[mask_cn_vacio, 'CLAIM NUMBER'] = (
            df_update_db.loc[mask_cn_vacio, 'KEY LOB']
            .str.split('-', n=1).str[0]
            .str.lstrip("'")
        )
        print(f'    CLAIM NUMBER: {mask_cn_vacio.sum()} extraidos desde KEY LOB')

    # LoB-Inward y COVER: extraer del KEY LOB respetando el pipeline de normalización
    # KEY LOB LoB part → cover_map → COVER canónico → maplobinward → LoB-Inward final
    lob_raw = (
        df_update_db.loc[new_claims_mask, 'KEY LOB']
        .str.split('-', n=1).str[1]   # parte después del primer '-'
        .str.strip()
        .str.upper()
    )
    # Paso 1: normalizar con cover_map para obtener COVER canónico
    cover_desde_keylob = lob_raw.map(lambda v: cover_map.get(v, v))
    # Paso 2: aplicar maplobinward para obtener LoB-Inward final
    lob_desde_keylob = cover_desde_keylob.map(lambda v: maplobinward.get(v, v))

    # Rellenar LoB-Inward vacío
    mask_lob_vacio = new_claims_mask & (
        df_update_db['LoB-Inward'].isna()
        | df_update_db['LoB-Inward'].astype(str).str.strip().isin(['', 'nan', 'NAN', 'None'])
    )
    if mask_lob_vacio.any():
        df_update_db.loc[mask_lob_vacio, 'LoB-Inward'] = lob_desde_keylob[mask_lob_vacio]
        print(f'    LoB-Inward: {mask_lob_vacio.sum()} vacios rellenados desde KEY LOB')

    # Rellenar COVER vacío con el valor canónico (cover_map aplicado al LoB del KEY LOB)
    if 'COVER' in df_update_db.columns:
        mask_cover_vacio = new_claims_mask & (
            df_update_db['COVER'].isna()
            | df_update_db['COVER'].astype(str).str.strip().isin(['', 'nan', 'NAN', 'None', 'NO ESPECIFICADO'])
        )
        if mask_cover_vacio.any():
            df_update_db.loc[mask_cover_vacio, 'COVER'] = cover_desde_keylob[mask_cover_vacio]
            print(f'    COVER: {mask_cover_vacio.sum()} vacios rellenados desde KEY LOB via cover_map')


    meta_mapping = {
        'INWARD POLICY N°':         'INWARD POLICY N°',
        'DATE OF LOSS':             'DATE OF LOSS',
        'DATE OF REPORT':           'DATE OF REPORT',
        'POLICY PERIOD START DATE':  'POLICY PERIOD START DATE',
        'POLICY PERIOD END DATE':    'POLICY PERIOD END DATE',
        'DESCRIPTION':              'DESCRIPTION'
    }
    for meta_col, db_col in meta_mapping.items():
            if meta_col in meta_indexed.columns and db_col in df_update_db.columns:
                
                # Creamos la serie mapeada SIN el .values
                valores_mapeados = meta_key_series.map(meta_indexed[meta_col])
                
                df_update_db.loc[new_claims_mask, db_col] = (
                    df_update_db.loc[new_claims_mask, db_col]
                    .fillna(valores_mapeados) # <--- Pasamos la Serie directamente
                )
                print(f"    Relleno: {db_col} desde contabilidad")
            elif db_col not in df_update_db.columns:
                print(f"    AVISO: {db_col} no existe en df_update_db")
    # --- Fin relleno metadatos ---

    # GROSS RESERVE y DEDUCTIBLE con 0 (contabilidad no los tiene)
    df_update_db.loc[new_claims_mask, f'GROSS RESERVE {AñoMes}'] = df_update_db.loc[new_claims_mask, f'GROSS RESERVE {AñoMes}'].fillna(0)
    df_update_db.loc[new_claims_mask, f'DEDUCTIBLE {AñoMes}'] = df_update_db.loc[new_claims_mask, f'DEDUCTIBLE {AñoMes}'].fillna(0)

    # Campos anteriores también con 0
    df_update_db.loc[new_claims_mask, f'GROSS RESERVE {AñoMes_ant}'] = df_update_db.loc[new_claims_mask, f'GROSS RESERVE {AñoMes_ant}'].fillna(0)
    df_update_db.loc[new_claims_mask, f'DEDUCTIBLE {AñoMes_ant}'] = df_update_db.loc[new_claims_mask, f'DEDUCTIBLE {AñoMes_ant}'].fillna(0)
    
    # Relleno de columnas de salvage y otros acumulados
    other_cols_to_fill = [
        f'Cumulative EXPENSES PAID {AñoMes_ant}',
        f'Cumulative CLAIMS PAID {AñoMes_ant}',
        f'Cumulative VALUATION EXPENSES {AñoMes_ant}',
        'Cumulative SALVAGE',
        f'OSLR Inward {AñoMes_ant}'
    ]
    
    for col in other_cols_to_fill:
        if col in df_update_db.columns:
            df_update_db.loc[new_claims_mask, col] = df_update_db.loc[new_claims_mask, col].fillna(0)
    
    # Rellenar campos de texto con valores por defecto
    text_defaults = {
        'STATUS': 'P',  # Pendiente
        'PAYMENT STATUS': 'OPEN',
        'Monthly': 0,
        'NOTAS': 'Siniestro agregado desde contabilidad'
    }
    
    for col, default_val in text_defaults.items():
        if col in df_update_db.columns:
            df_update_db.loc[new_claims_mask, col] = df_update_db.loc[new_claims_mask, col].fillna(default_val)
    
    print(f"  ✓ Registros nuevos rellenados correctamente")
else:
    print(f"  No hay registros nuevos para rellenar")

# =============================================================================
# PASO 6.7: FIX #3 — DETECCIÓN Y MANEJO DE CLAIMS HUÉRFANOS
# =============================================================================
print('\n🔍 DETECCIÓN DE CLAIMS HUÉRFANOS (Payments sin match en BDX):')

# Claims que solo vienen de contabilidad: tienen pagos pero no datos de BDX
mask_orphan = (
    (df_update_db['FLAG CONTA'] == 1) & 
    (
        df_update_db['INWARD POLICY N°'].isna() | 
        (df_update_db['INWARD POLICY N°'].astype(str).str.strip() == '') |
        (df_update_db['INWARD POLICY N°'].astype(str).str.strip() == 'nan')
    )
)

orphan_claims = df_update_db[mask_orphan].copy()

if len(orphan_claims) > 0:
    print(f'  ⚠️  {len(orphan_claims)} claims de contabilidad sin match en BDX:')
    for idx, row in orphan_claims.iterrows():
        key = row['KEY LOB']
        claim_num = key.split('-')[0] if '-' in str(key) else str(key)
        exp = row.get(f'Cumulative EXPENSES PAID {AñoMes}', 0)
        cl = row.get(f'Cumulative CLAIMS PAID {AñoMes}', 0)
        va = row.get(f'Cumulative VALUATION EXPENSES {AñoMes}', 0)
        print(f'    - {key}: Expenses=${exp:,.2f}, Claims=${cl:,.2f}, Valuation=${va:,.2f}')
    
    # Intentar match flexible: buscar por CLAIM NUMBER sin LoB
    print('\n  🔗 Intentando match flexible por CLAIM NUMBER:')
    for idx in orphan_claims.index:
        key_lob = str(df_update_db.loc[idx, 'KEY LOB'])
        claim_num = key_lob.split('-')[0] if '-' in key_lob else key_lob
        
        # Buscar registros con el mismo CLAIM NUMBER que sí tengan datos de BDX
        possible = df_update_db[
            (df_update_db['CLAIM NUMBER'].astype(str) == claim_num) & 
            (df_update_db.index != idx) &
            (df_update_db['INWARD POLICY N°'].notna()) &
            (df_update_db['INWARD POLICY N°'].astype(str).str.strip() != '') &
            (df_update_db['INWARD POLICY N°'].astype(str).str.strip() != 'nan')
        ]
        
        if len(possible) > 0:
            source = possible.iloc[0]
            cols_to_copy = ['INWARD POLICY N°', 'SUBSIDIARY', 'DATE OF LOSS', 
                          'DATE OF REPORT', 'INWARD POLICY',
                          'POLICY PERIOD START DATE', 'POLICY PERIOD END DATE']
            for col in cols_to_copy:
                if col in df_update_db.columns and col in source.index:
                    df_update_db.loc[idx, col] = source[col]
            print(f'    ✓ {key_lob}: datos copiados de registro existente (Póliza: {source.get("INWARD POLICY N°", "?")})')
        else:
            print(f'    ✗ {key_lob}: SIN MATCH — requiere revisión manual')
    
    # Exportar reporte de huérfanos
    try:
        orphan_report = df_update_db[mask_orphan][['KEY LOB', 
            f'Cumulative EXPENSES PAID {AñoMes}',
            f'Cumulative CLAIMS PAID {AñoMes}', 
            f'Cumulative VALUATION EXPENSES {AñoMes}']].copy()
        orphan_report.to_excel(f'{ruta_procesados}/{AñoMes}/CLAIMS_HUERFANOS_{AñoMes}.xlsx', index=False)
        print(f'\n  📄 Reporte exportado: CLAIMS_HUERFANOS_{AñoMes}.xlsx')
    except Exception as e:
        print(f'\n  ⚠️  No se pudo exportar reporte de huérfanos: {e}')
else:
    print('  ✓ Todos los claims de contabilidad tienen match en BDX')

# =============================================================================
# PASO 6.9: LIMPIEZA GENERAL — rellenar LoB-Inward vacíos en TODOS los registros
# Cubre casos no capturados por PASO 6.5 (BDX con LoB vacío, orphans, etc.)
print('\n🔧 LIMPIEZA GENERAL DE LoB-Inward:')

mask_lob_global = (
    df_update_db['LoB-Inward'].isna()
    | df_update_db['LoB-Inward'].astype(str).str.strip().isin(['', 'nan', 'NAN', 'None', 'NO ESPECIFICADO'])
)

if mask_lob_global.any():
    print(f'  Registros con LoB-Inward vacío: {mask_lob_global.sum()}')

    # Intento 1: derivar de COVER
    cover_col_available = 'COVER' if 'COVER' in df_update_db.columns else ('COVERAGE' if 'COVERAGE' in df_update_db.columns else None)
    if cover_col_available:
        lob_desde_cover = (
            df_update_db.loc[mask_lob_global, cover_col_available]
            .astype(str).str.strip().str.upper()
            .replace(cover_map)
            .map(maplobinward)
        )
        filled_cover = lob_desde_cover.notna() & (lob_desde_cover.astype(str).str.strip() != '')
        df_update_db.loc[mask_lob_global & filled_cover.reindex(df_update_db.index, fill_value=False), 'LoB-Inward'] = \
            lob_desde_cover[filled_cover]
        mask_lob_global = (
            df_update_db['LoB-Inward'].isna()
            | df_update_db['LoB-Inward'].astype(str).str.strip().isin(['', 'nan', 'NAN', 'None', 'NO ESPECIFICADO'])
        )
        print(f'  Tras derivar de COVER: {mask_lob_global.sum()} siguen vacíos')

    # Intento 2: extraer del KEY LOB como último recurso
    if mask_lob_global.any():
        lob_desde_keylob = (
            df_update_db.loc[mask_lob_global, 'KEY LOB']
            .str.split('-', n=1).str[1]
            .str.strip().str.upper()
            .map(lambda v: cover_map.get(str(v), str(v)) if pd.notna(v) else v)
            .map(lambda v: maplobinward.get(str(v), str(v)) if pd.notna(v) else v)
        )
        filled_keylob = lob_desde_keylob.notna() & ~lob_desde_keylob.astype(str).str.strip().isin(['', 'nan', 'NAN', 'None'])
        df_update_db.loc[mask_lob_global & filled_keylob.reindex(df_update_db.index, fill_value=False), 'LoB-Inward'] = \
            lob_desde_keylob[filled_keylob]
        mask_lob_global = (
            df_update_db['LoB-Inward'].isna()
            | df_update_db['LoB-Inward'].astype(str).str.strip().isin(['', 'nan', 'NAN', 'None', 'NO ESPECIFICADO'])
        )
        print(f'  Tras extraer de KEY LOB: {mask_lob_global.sum()} siguen vacíos (requieren revisión manual)')
else:
    print('  ✓ No hay LoB-Inward vacíos')

# PASO 7: LIMPIEZA DE OTRAS COLUMNAS
# =============================================================================
print('\n🧹 LIMPIEZA DE OTRAS COLUMNAS:')

# Eliminar versiones _y/_old/_new de CLAIM NUMBER y LoB-Inward
cols_patterns_to_clean = ['CLAIM NUMBER', 'LoB-Inward']
for pattern in cols_patterns_to_clean:
    cols_to_drop = [col for col in df_update_db.columns 
                    if col.startswith(pattern) and (col.endswith('_y') or 
                                                     col.endswith('_old') or 
                                                     col.endswith('_new'))]
    if cols_to_drop:
        df_update_db = df_update_db.drop(columns=cols_to_drop)
        print(f"  ✓ Eliminadas: {cols_to_drop}")

# Renombrar si existen versiones _x/_old
for pattern in cols_patterns_to_clean:
    col_x = f'{pattern}_x'
    col_old = f'{pattern}_old'
    
    if col_x in df_update_db.columns:
        df_update_db = df_update_db.rename(columns={col_x: pattern})
        print(f"  ✓ Renombrado: {col_x} → {pattern}")
    elif col_old in df_update_db.columns:
        df_update_db = df_update_db.rename(columns={col_old: pattern})
        print(f"  ✓ Renombrado: {col_old} → {pattern}")


# =============================================================================
# PASO 8: VALIDACIÓN FINAL
# =============================================================================
print('\n' + '=' * 95)
print('RESULTADO FINAL DEL CRUCE:')
print(f'📊 Registros en df_update_db: {len(df_update_db)}')

# Verificar columnas críticas
columnas_criticas = ['KEY LOB', 'FLAG CONTA', 'CLAIM NUMBER', 'LoB-Inward']
print('\n✅ COLUMNAS CRÍTICAS:')
for col in columnas_criticas:
    existe = col in df_update_db.columns
    print(f"  {col}: {'✓ Existe' if existe else '✗ NO EXISTE'}")

# Estadísticas de FLAG CONTA
if 'FLAG CONTA' in df_update_db.columns:
    registros_cruzados = df_update_db['FLAG CONTA'].sum()
    registros_sin_match = (df_update_db['FLAG CONTA'] == 0).sum()
    tasa_match = (registros_cruzados / len(df_update_db)) * 100 if len(df_update_db) > 0 else 0
    
    print(f'\n📈 ESTADÍSTICAS DE CRUCE:')
    print(f'  Registros que cruzaron: {registros_cruzados:,.0f}')
    print(f'  Registros sin match: {registros_sin_match:,.0f}')
    print(f'  Tasa de match: {tasa_match:.2f}%')
else:
    print('\n❌ ERROR: FLAG CONTA no existe en el resultado final')
    print('   Columnas disponibles:')
    print(f'   {df_update_db.columns.tolist()[:20]}...')



Comenzamos con el cruce entre la base actualizable del mes y la información contable del mes
Tenemos 4255 registros en la base actualizable del mes
Tenemos 46 registros en la base contable del mes

🔍 COLUMNAS EN df_update_db ANTES DE LIMPIEZA:
[]

🧹 LIMPIEZA DE COLUMNAS _y:
No hay columnas _y para eliminar

🔧 RENOMBRANDO COLUMNAS _x:
Columnas _x encontradas: []
No hay columnas _x para renombrar

🔍 VERIFICACIÓN POST-LIMPIEZA:
¿FLAG CONTA existe en df_update_db? False

📊 PREPARACIÓN DEL MERGE:
✅ Columnas disponibles en df_conta_final: ['KEY LOB', 'FLAG CONTA', 'Cumulative EXPENSES PAID 202509', 'Cumulative CLAIMS PAID 202509', 'Cumulative VALUATION EXPENSES 202509']

🔗 REALIZANDO MERGE:
Registros antes del merge: 4255
FLAG CONTA existía antes del merge: False

📊 INFO PRE-MERGE:
  Registros en df_update_db: 4255
  Registros en df_conta_final: 46
  KEY LOB en df_conta_final: ['108638/2018-JACK-UPS(DAÑO FISICO)', '110697/2018-PANDI', '112224/2017-CASCO Y MAQUINARIA', '1558/2024-CARGO', '21/

## 6. Creation of formulated variables

In [357]:
# =============================================================================
# Section 6: Creation of formulated variables
# =============================================================================

# ======= Numerical variables ======= #
# == Exceptions == #
# OSLR
# List of records for wich we use the Gross Reserve and Deductible from previous month
list_ded_reserve = df_update_db[
(df_update_db['NOTAS'] != '')| # All the records that have a commment at this point
(df_update_db['Monthly'] == 1) # All the records that are monthly 
]['CLAIM NUMBER'].unique()

# Initialize the list of claim numbers with incidences during the month
list_exceptions_monthly = df_update_db[
(df_update_db['NOTAS'] != '') & # The claim number that has a comment in 'NOTAS'
(~df_update_db['NOTAS'].isin(['LEGACY INFO', 'LEGACY POLICY'])) # but the comment is different from 'LEGACY INFO', 'LEGACY POLICY'
]['CLAIM NUMBER'].unique()  

# Fill the current values of gross reserve and deductible with the values from previous month 
for col in [f'GROSS RESERVE {AñoMes_ant}', f'DEDUCTIBLE {AñoMes_ant}']:
    target_col = col.replace(str(AñoMes_ant), str(AñoMes))  # Update target column name dynamically
    df_update_db.loc[
        df_update_db['CLAIM NUMBER'].isin(list_ded_reserve),  # If the record is in the list 'list_ded_reserve'
        target_col                                                  # Column to be updated
    ] = df_update_db[col]                                           # Value from the previous month

import re
# Expresión regular para eliminar caracteres ilegales para Excel
def clean_excel_string(s):
    if isinstance(s, str):
        # Quita caracteres no imprimibles ni válidos en XML
        return re.sub(r'[\x00-\x1F\x7F-\x9F]', '', s)
    return s

# Identificar columnas numéricas
numeric_cols = df_update_db.select_dtypes(include=['number']).columns.tolist()
string_cols = df_update_db.select_dtypes(include=['object']).columns.tolist()

print(f"Columnas numéricas: {len(numeric_cols)}")
print(f"Columnas de texto: {len(string_cols)}")

# Aplicar limpieza SOLO a columnas de texto
for col in string_cols:
    df_update_db[col] = df_update_db[col].apply(clean_excel_string)

print("✓ Limpieza completada solo en columnas de texto")

# Debugg
df_update_db.to_excel(f'{ruta_procesados}/{AñoMes}/pruebas/actuarial.xlsx', index=False)

Columnas numéricas: 16
Columnas de texto: 21


C:\Users\IKAL14\AppData\Local\Temp\ipykernel_8984\2518008914.py:38: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_cols = df_update_db.select_dtypes(include=['object']).columns.tolist()


✓ Limpieza completada solo en columnas de texto


In [358]:
# =============================================================================
# Section 5.5: FALLBACK LOGIC 
# (Mantener valores anteriores cuando el actual es 0)
# ¡IMPORTANTE! Esto DEBE estar ANTES de Section 6 (OSLR calculation)
# =============================================================================

print("\n" + "="*95)
print("APLICANDO FALLBACK: Mantener valores anteriores cuando el actual es 0")
print("="*95)

# Para GROSS RESERVE
# FIX #5: Verificar que el claim NO esté cerrado antes de restaurar GR anterior.
# Un claim cerrado con GR=0 es correcto — no se debe restaurar el valor anterior.
status_col = 'STATUS' if 'STATUS' in df_update_db.columns else None
if status_col:
    mask_gr = (
        (df_update_db[f'GROSS RESERVE {AñoMes}'] == 0) & 
        (df_update_db[f'GROSS RESERVE {AñoMes_ant}'] > 0) &
        (~df_update_db[status_col].astype(str).str.upper().isin(['CLOSED', 'C', 'CERRADO']))
    )
else:
    # Si no hay columna STATUS, mantener comportamiento original
    mask_gr = (df_update_db[f'GROSS RESERVE {AñoMes}'] == 0) & (df_update_db[f'GROSS RESERVE {AñoMes_ant}'] > 0)
df_update_db.loc[mask_gr, f'GROSS RESERVE {AñoMes}'] = df_update_db.loc[mask_gr, f'GROSS RESERVE {AñoMes_ant}']

# Para DEDUCTIBLE
mask_ded = (df_update_db[f'DEDUCTIBLE {AñoMes}'] == 0) & (df_update_db[f'DEDUCTIBLE {AñoMes_ant}'] > 0)
df_update_db.loc[mask_ded, f'DEDUCTIBLE {AñoMes}'] = df_update_db.loc[mask_ded, f'DEDUCTIBLE {AñoMes_ant}']

# Log de registros ajustados
registros_gr_ajustados = mask_gr.sum()
registros_ded_ajustados = mask_ded.sum()

print(f"✓ Registros donde se mantuvo GROSS RESERVE anterior: {registros_gr_ajustados}")
print(f"✓ Registros donde se mantuvo DEDUCTIBLE anterior: {registros_ded_ajustados}")

# Verificación de siniestros específicos
claim_series = (df_update_db['CLAIM NUMBER'].astype("string").fillna(""))

siniestros_verificar = ['324372640000425', '324372640000403']

for claim_check_id in siniestros_verificar:
    mask = claim_series == claim_check_id
    if mask.any():
        claim_check = df_update_db.loc[mask]
        print(f"\n  Siniestro {claim_check_id}:")
        print(f"    GROSS RESERVE ({AñoMes}): {claim_check[f'GROSS RESERVE {AñoMes}'].iloc[0]}")
        print(f"    DEDUCTIBLE ({AñoMes}): {claim_check[f'DEDUCTIBLE {AñoMes}'].iloc[0]}")
    else:
        print(f"\n  ⚠️ Siniestro {claim_check_id} no encontrado")

print("="*95 + "\n")



APLICANDO FALLBACK: Mantener valores anteriores cuando el actual es 0
✓ Registros donde se mantuvo GROSS RESERVE anterior: 651
✓ Registros donde se mantuvo DEDUCTIBLE anterior: 576

  Siniestro 324372640000425:
    GROSS RESERVE (202509): 30000.0
    DEDUCTIBLE (202509): 10000.0

  Siniestro 324372640000403:
    GROSS RESERVE (202509): 30000.0
    DEDUCTIBLE (202509): 10000.0



In [359]:
# ============== Payments columns ============== #
# == Creation of formulated columns == #
#
df_update_db['Cumulative EXPENSES PAID'] = df_update_db[f'Cumulative EXPENSES PAID {AñoMes_ant}'] + df_update_db[f'Cumulative EXPENSES PAID {AñoMes}']
#
df_update_db['Cumulative VALUATION EXPENSES'] = df_update_db[f'Cumulative VALUATION EXPENSES {AñoMes_ant}'] + df_update_db[f'Cumulative VALUATION EXPENSES {AñoMes}']
#
df_update_db['Cumulative CLAIMS PAID'] = df_update_db[f'Cumulative CLAIMS PAID {AñoMes_ant}'] + df_update_db[f'Cumulative CLAIMS PAID {AñoMes}']
#
df_update_db['Total Claims Paid Inward'] = df_update_db['Cumulative EXPENSES PAID'] + df_update_db['Cumulative VALUATION EXPENSES'] + df_update_db['Cumulative CLAIMS PAID'] + df_update_db['Cumulative SALVAGE']
#
df_update_db['Total Claims Paid no Alae'] = df_update_db['Total Claims Paid Inward'] - df_update_db['Cumulative EXPENSES PAID']



# ============== OSLR Calculation ============== #

# Normal calculation
df_update_db[f'OSLR Inward {AñoMes}'] = np.maximum(
    np.maximum(df_update_db[f'GROSS RESERVE {AñoMes}'] - df_update_db[f'DEDUCTIBLE {AñoMes}'], 0) - df_update_db['Cumulative CLAIMS PAID'], 0)

# 🆕 Ajuste: Valores menores a 1 se ponen en 0
df_update_db.loc[df_update_db[f'OSLR Inward {AñoMes}'] < 1, f'OSLR Inward {AñoMes}'] = 0

# For legacy records we set the value from the previous month
df_update_db.loc[df_update_db['NOTAS'].isin(['LEGACY INFO', 'LEGACY POLICY']), f'OSLR Inward {AñoMes}'] = df_update_db[f'OSLR Inward {AñoMes_ant}']

# ====================== Handmade Adjustments ====================== #
# This section is made after the review of the recods #
# Set OSLR Inward to 0 for specific CLAIM NUMBERS
df_update_db.loc[
    df_update_db['CLAIM NUMBER'].isin([
    '4431/2023',
    '1560/2024',
    '1563/2024',
    '4478/2024',
    '4801/2024',
    '4803/2024',
    '5704/2024',
    '6399/2024',
    '6400/2024',
    '161/2025',
    '594/2025'
    ]),
    f'OSLR Inward {AñoMes}'
] = 0

#  ================================================================== #
#df_update_db.loc[
#    (df_update_db[f'OSLR Inward {AñoMes_ant}'] == 0) & (df_update_db['STATUS'].isin(['P','p']) ),
#    f'OSLR Inward {AñoMes}'
#] = 0


# Additional corrections
# ==  Correction if the claim is already closed but it had and incident during the month == #
# Conditions for this correction

mask = (
    #(df_update_db[f'OSLR Inward {AñoMes_ant}'] == 0) &  # If in the previous month its OSLR was 0
    ((df_update_db[f'OSLR Inward {AñoMes}'] - df_update_db[f'OSLR Inward {AñoMes_ant}']).fillna(0).round(2) != 0) & # Si la diferencia en OSLR es distinta de 0
    (df_update_db['CLAIM NUMBER'].isin(list_exceptions_monthly)) &  # If the claim is in the list of monthly incidences
    (df_update_db['FLAG CONTA'] == 0) & # If the claim doesnt have account movements during the month
    (df_update_db['Monthly'] != 1) #&  This doesn´t apply for monthly records
    #(~df_update_db['CLAIM NUMBER'].isin(['2893/2024', '3191/2024','3455/2024','3611/2024','4068/2024','1721/2024']) ) # Records added at the end
)

# Apply the correction
df_update_db.loc[mask, f'OSLR Inward {AñoMes}'] = 0

# Print the summary of changes
updated_count = mask.sum()
print(f"Number of records updated to 0 in OSLR Inward {AñoMes}: {updated_count}")

# Print the CLAIM NUMBER of updated records
if updated_count > 0:
    updated_claims = df_update_db.loc[mask, 'CLAIM NUMBER'].tolist()
    print("The following CLAIM NUMBERS were updated:")
    print(updated_claims)
else:
    print("No records were updated.")

# ============== INCURRED ============== #
# 'Inward Incurred'
df_update_db['Inward Incurred'] = df_update_db['Total Claims Paid Inward'] + df_update_db[f'OSLR Inward {AñoMes}']
# 'Inward Incurred no Alae'
df_update_db['Inward Incurred no Alae'] = df_update_db['Inward Incurred'] - df_update_db['Cumulative EXPENSES PAID']

# ======= Other variables ======= #
df_update_db['YEAR OF THE RESERVE'] = date_start_AñoMes.year
df_update_db['ACCIDENT YEAR'] = df_update_db['DATE OF LOSS'].dt.year
df_update_db['CLAIMS-ID'] = df_update_db['CLAIM NUMBER ORIGINAL'].astype(str) + df_update_db['LoB-Inward BDX'].astype(str) 
df_update_db['ATTRITIONAL/LARGE'] = 'ATTRITIONAL'
df_update_db['INWARD POLICY'] = '02-Marine'
df_update_db['StandRe LoB'] = '03-MAT'
df_update_db['StandRe detailed LoB'] = 'Combined marine' 
df_update_db['UW Year'] = np.where(
    df_update_db['DATE OF LOSS'].dt.month < 7,
    df_update_db['DATE OF LOSS'].dt.year - 1,
    df_update_db['DATE OF LOSS'].dt.year
)

#df_update_db.to_excel(f'C:/Users/IKAL14/Desktop/CALCULOS.xlsx', index= False )

Number of records updated to 0 in OSLR Inward 202509: 2
The following CLAIM NUMBERS were updated:
['322361640000001', '351003048179']


## 7. OSLR and Dates Validation

In [360]:
# =============================================================================
# Section 7: OSLR and Dates Validation
# =============================================================================

print('\n' + '='*95)
print('SECCIÓN 7: VALIDACIONES DE OSLR Y FECHAS')
print('='*95)

# Identificar registros nuevos (solo de contabilidad) ANTES de hacer validaciones
# Estos NO deben ser validados de forma estricta
new_records_mask = (df_update_db['FLAG CONTA'] == 1) & (df_update_db['NOTAS'] == 'Siniestro agregado desde contabilidad')
new_records_count = new_records_mask.sum()

print(f"\n📌 Registros nuevos (excluidos de validaciones estrictas): {new_records_count}")
if new_records_count > 0:
    print(f"   Estos registros se mantienen sin validación de fechas/OSLR")
    new_records_list = df_update_db.loc[new_records_mask, 'KEY LOB'].unique()
    for claim_key in new_records_list:
        print(f"   - {claim_key}")

# ======= Validations ======= #

# == Validation of 'OSLR Inward == #
df_update_db['DIF OSLR'] = (df_update_db[f'OSLR Inward {AñoMes}'] - df_update_db[f'OSLR Inward {AñoMes_ant}']).fillna(0).round(2)

# Filter the records that meet the following conditions
# EXCLUIR registros nuevos de esta validación
df_errors_oslr = df_update_db[
    (df_update_db['DIF OSLR'].round() != 0) & # Exist a difference of OSLR within months
    (df_update_db['FLAG CONTA'] == 0) & # The records doesn't have accounting movements in the month of process
    (df_update_db['Monthly'] == 0) & # The record is not a new claim
    ~(((df_update_db[f'GROSS RESERVE {AñoMes}'] - df_update_db[f'GROSS RESERVE {AñoMes_ant}']) != 0) | #Excludes records that had 
     ((df_update_db[f'DEDUCTIBLE {AñoMes}'] - df_update_db[f'DEDUCTIBLE {AñoMes_ant}']) != 0)) &  # a change in Gross Reserve or Deductible from the previous month.
    ~new_records_mask  # EXCLUIR registros nuevos
]

# == Validation of Dates == #
# EXCLUIR registros nuevos (que no tienen fechas)
df_errors_dates = df_update_db[
    (
        (df_update_db['DATE OF LOSS'] < df_update_db['POLICY PERIOD START DATE']) |
        (df_update_db['DATE OF LOSS'] > df_update_db['POLICY PERIOD END DATE']) |
        ((df_update_db['DATE OF LOSS'] > df_update_db['DATE OF REPORT']) &
         (df_update_db['STATUS'] != 'C'))
    ) &
    ~new_records_mask  # EXCLUIR registros nuevos (aplica a TODAS las condiciones)
].copy()

# Sanitize status
df_errors_dates.loc[:, 'STATUS'] = df_errors_dates['STATUS'].astype(str).str.strip().str.capitalize()

# Drop canceled and NaN records from this filtered DataFrame
df_errors_dates = df_errors_dates[~df_errors_dates['STATUS'].isin(['C','Nan',''])]

# Report
print('==============================================================================')
if not df_errors_dates.empty:
    print('Los registros con errores en fechas son los siguientes:')
    for index, row in df_errors_dates.iterrows():
        print(
            f"\nCLAIM NUMBER: {row['CLAIM NUMBER']}\n"
            f"  DATE OF LOSS          : {row['DATE OF LOSS']}\n"
            f"  DATE OF REPORT        : {row['DATE OF REPORT']}\n"
            f"  POLICY PERIOD START   : {row['POLICY PERIOD START DATE']}\n"
            f"  POLICY PERIOD END     : {row['POLICY PERIOD END DATE']}\n"
            f"  STATUS                : {row['STATUS']}\n"
        )
else:
    print("No se encontraron errores en fechas.")
print('==============================================================================')

# Final selection of columns
df_errors_dates = df_errors_dates[['CLAIM NUMBER ORIGINAL','DATE OF LOSS','DATE OF REPORT',
                                   'INWARD POLICY N°','LoB-Inward','POLICY PERIOD START DATE',
                                   'POLICY PERIOD END DATE','STATUS', 'NOTAS']]


# Create the error column according to the type of error
# Define the conditions and its value
conditions = [
    df_errors_dates['DATE OF LOSS'] < df_errors_dates['POLICY PERIOD START DATE'],
    df_errors_dates['DATE OF LOSS'] > df_errors_dates['POLICY PERIOD END DATE'],
    df_errors_dates['DATE OF LOSS'] > df_errors_dates['DATE OF REPORT']
]

# Define the choices
choices = ['DOL antes vigencia', 'DOL después vigencia', 'DOL después reporte']

# Create the column 'ERROR' based on the conditions
df_errors_dates['ERROR'] = np.select(conditions, choices, default='Sin error')
# Exportof the file
df_errors_dates.to_excel(f'{ruta_incidencias}/Incidencias_fechas.xlsx' ,index= False)
print('Se exportó correctamente el archivo de incidencias de fechas')
print('==============================================================================')

# LoB normalization
# Convert the relevant columns to string type
df_update_db[['LoB-Inward BDX', 'LoB-Inward','CLAIM NUMBER ORIGINAL']] = df_update_db[['LoB-Inward BDX', 'LoB-Inward','CLAIM NUMBER ORIGINAL']].astype(str)

# Replace 'nan' strings with empty strings in 'LoB-Inward BDX'
df_update_db['LoB-Inward BDX'] = df_update_db['LoB-Inward BDX'].replace('nan', '')

# Fill empty values in 'LoB-Inward BDX' with the values from 'LoB-Inward'
df_update_db.loc[df_update_db['LoB-Inward BDX'] == '', 'LoB-Inward BDX'] = df_update_db['LoB-Inward']
df_update_db['CLAIMS-ID'] = df_update_db['CLAIM NUMBER ORIGINAL'] + "-" + df_update_db['LoB-Inward BDX']

# CLAIMS-ID vacío → KEY LOB (igual que NB4 cell 10)
df_update_db['CLAIMS-ID'] = df_update_db['CLAIMS-ID'].fillna(df_update_db['KEY LOB'])
mask_claimsid_vacio = df_update_db['CLAIMS-ID'].astype(str).str.strip().isin(['', 'nan', 'None', 'nan-nan'])
if mask_claimsid_vacio.any():
    df_update_db.loc[mask_claimsid_vacio, 'CLAIMS-ID'] = df_update_db.loc[mask_claimsid_vacio, 'KEY LOB']
    print(f'CLAIMS-ID: {mask_claimsid_vacio.sum()} vacios rellenados con KEY LOB')


# Debugg
#df_update_db.to_excel(f'{ruta_procesados}/pruebas/actuarial.xlsx', index= False )


SECCIÓN 7: VALIDACIONES DE OSLR Y FECHAS

📌 Registros nuevos (excluidos de validaciones estrictas): 2
   Estos registros se mantienen sin validación de fechas/OSLR
   - 324392640000098-CARGO
   - 6490/2022-CARGO
Los registros con errores en fechas son los siguientes:

CLAIM NUMBER: 323361640000029
  DATE OF LOSS          : 2023-03-08 00:00:00
  DATE OF REPORT        : 2023-05-30 00:00:00
  POLICY PERIOD START   : 2021-02-20 00:00:00
  POLICY PERIOD END     : 2023-02-20 00:00:00
  STATUS                : P

Se exportó correctamente el archivo de incidencias de fechas


## 8. Final Adjustments

In [361]:
# =============================================================================
# Section 8: Final adjustments
# =============================================================================

print('\n' + '='*95)
print('SECCIÓN 8: AJUSTES FINALES')
print('='*95)

# Verificar que las columnas necesarias existan ANTES de hacer operaciones
print('\n🔍 Verificando columnas críticas para final adjustment:')

# Rellenar cualquier NaN restante en columnas críticas
critical_cols = [
    f'GROSS RESERVE {AñoMes}', f'DEDUCTIBLE {AñoMes}',
    'LoB-Inward BDX', 'CLAIM NUMBER ORIGINAL',
    'Cumulative SALVAGE', 'Cumulative EXPENSES PAID', 'Cumulative VALUATION EXPENSES'
]

for col in critical_cols:
    if col in df_update_db.columns:
        nulos = df_update_db[col].isna().sum()
        if nulos > 0:
            # Rellenar NaN con 0 para columnas numéricas, vacío para texto
            if df_update_db[col].dtype in ['float64', 'int64']:
                df_update_db[col] = df_update_db[col].fillna(0)
            else:
                df_update_db[col] = df_update_db[col].fillna('')
            print(f"  ✓ {col}: {nulos} valores rellenados")

df_update_db.drop(columns=['LoB-Inward','CLAIM NUMBER'], inplace= True)
df_update_db.rename(columns={'LoB-Inward BDX': 'LoB-Inward','CLAIM NUMBER ORIGINAL' :'CLAIM NUMBER',
                             f'OSLR Inward {AñoMes}': 'OSLR Inward',
                             f'GROSS RESERVE {AñoMes}': f'GROSS RESERVE',
                             f'DEDUCTIBLE {AñoMes}': f'DEDUCTIBLE'
                             }, inplace= True)

# Rename of  policies 
df_update_db.loc[df_update_db['INWARD POLICY N°'] == '90600-323484','INWARD POLICY N°'] = '90600 323484'
df_update_db.loc[df_update_db['INWARD POLICY N°'] == '90600-328256','INWARD POLICY N°'] = '90600 328256'

#LoB normalization
# LoB of legacy info
df_update_db.loc[df_update_db['LoB-Inward'] == 'CASCO', 'LoB-Inward'] = 'CASCO Y MAQUINARIA'

# Subsidiary Normalization
df_update_db['SUBSIDIARY'] = df_update_db['SUBSIDIARY'].str.strip() # Delete all the spaces at the end and beggining

# Normalize the 'SUBSIDIARY' column
df_update_db["SUBSIDIARY"] = (
df_update_db["SUBSIDIARY"]
    #todo en mayusculas
    .str.upper()
    #remueve la palabra pemex
    .str.replace(r"\bPEMEX\b", "", regex=True)
    #remueve espacios extras
    .str.replace(r"\s+", " ", regex=True)
    #elimina espacios al inicio y final
    .str.strip()
)
# Initialize a dictionary with the correct names
dict_subsidiaries = {
    'PEMEX Logistica':'LOGÍSTICA',
    'LOGISTICA':'LOGÍSTICA',
    'Logistica':'LOGÍSTICA',
    'Pemex Logística':'LOGÍSTICA',
    'LOG':'LOGÍSTICA'
}
# Apply the correction 
df_update_db['SUBSIDIARY'] = df_update_db['SUBSIDIARY'].replace(dict_subsidiaries)

#Cambiamos los valores de la columna Subsidiary
df_update_db['SUBSIDIARY'] = df_update_db['SUBSIDIARY'].replace({
'LOGÍSTICA':" DIRECCIÓN DE LOGÍSTICA Y SALVAGUARDIA ESTRATÉGICA ",
'REFINACIÓN':"  DIRECCIÓN DE PROCESOS INDUSTRIALES Y DIRECCIÓN DE TRANSFORMACIÓN ENERGÉTICA",
'PETROQUÍMICA':"  DIRECCIÓN DE PROCESOS INDUSTRIALES Y DIRECCIÓN DE TRANSFORMACIÓN ENERGÉTICA", 
'EXPLORACIÓN Y PRODUCCIÓN':"DIRECCIÓN DE EXPLORACIÓN Y EXTRACCIÓN", 
'PMI':"DIRECCIÓN DE EXPLORACIÓN Y EXTRACCIÓN",
'ETILENO':"  DIRECCIÓN DE PROCESOS INDUSTRIALES Y DIRECCIÓN DE TRANSFORMACIÓN ENERGÉTICA", 
'PERFORACIÓN Y SERVICIOS':"DIRECCIÓN DE EXPLORACIÓN Y EXTRACCIÓN",
'TRANSFORMACIÓN INDUSTRIAL':"  DIRECCIÓN DE PROCESOS INDUSTRIALES Y DIRECCIÓN DE TRANSFORMACIÓN ENERGÉTICA",
'REF':"  DIRECCIÓN DE PROCESOS INDUSTRIALES Y DIRECCIÓN DE TRANSFORMACIÓN ENERGÉTICA",
'PEP':"DIRECCIÓN DE EXPLORACIÓN Y EXTRACCIÓN",
'PTRI':"  DIRECCIÓN DE PROCESOS INDUSTRIALES Y DIRECCIÓN DE TRANSFORMACIÓN ENERGÉTICA",
'GAS Y PETROQUÍMICA BÁSICA':"  DIRECCIÓN DE PROCESOS INDUSTRIALES Y DIRECCIÓN DE TRANSFORMACIÓN ENERGÉTICA"})

# Fill NaN with 'NO ESPECIFICADO'
df_update_db['SUBSIDIARY'] = df_update_db['SUBSIDIARY'].fillna('NO ESPECIFICADO')

#Columns to drop
dropcolumns=['KEY DED', 'GROSS RESERVE 202509',
       'DEDUCTIBLE 202509', 'Cumulative EXPENSES PAID 202509',
       'Cumulative VALUATION EXPENSES 202509', 'Cumulative CLAIMS PAID 202509',
       'OSLR Inward 202509', 'CLAIM NUMBER ORIGINAL BDX',
       'POLICY PERIOD START DATE BDX', 'POLICY PERIOD END DATE BDX', 'Monthly',
       'CEDANT OBSERVATIONS', 'KOT OBSERVATIONS', 'GROSS RESERVE 202504',
       'DEDUCTIBLE 202504', 'Cumulative EXPENSES PAID 202504',
       'Cumulative VALUATION EXPENSES 202504', 'Cumulative CLAIMS PAID 202504',
       'OSLR Inward 202504', 'KEY LOB', 'DIF OSLR',
       'Cumulative EXPENSES PAID 202510', 'Cumulative CLAIMS PAID 202510',
       'Cumulative VALUATION EXPENSES 202510', 'FLAG CONTA']

# Final order
list_columns_order = [
'YEAR OF THE RESERVE','ACCIDENT YEAR','CLAIMS-ID','CLAIM NUMBER','DATE OF LOSS','DATE OF REPORT','ATTRITIONAL/LARGE',	
'INWARD POLICY','INWARD POLICY N°','POLICY PERIOD START DATE','POLICY PERIOD END DATE','SUBSIDIARY','StandRe LoB',
'StandRe detailed LoB','LoB-Inward','COVER','COVERAGE','STATUS','REF. PEMEX','LOCATION','DESCRIPTION' ,
'Cumulative SALVAGE','Cumulative EXPENSES PAID','Cumulative VALUATION EXPENSES','Cumulative CLAIMS PAID',
'Total Claims Paid Inward','Total Claims Paid no Alae','OSLR Inward','Inward Incurred',
'Inward Incurred no Alae',f'GROSS RESERVE',f'DEDUCTIBLE','KOT SHARE','UW Year','Nat Cat', 'KEY LOB','FLAG CONTA','NOTAS'
] #,'DIF OSLR', 'FLAG CONTA'

# Filtrar solo las columnas que existen en el dataframe
list_columns_order_final = [col for col in list_columns_order if col in df_update_db.columns]

# Agregar las columnas restantes que no estén en el orden especificado
cols_faltantes = [col for col in df_update_db.columns if col not in list_columns_order_final]
list_columns_order_final.extend(cols_faltantes)

print(f'\n✓ Reordenando columnas (Orden final: {len(list_columns_order_final)} columnas)')
df_update_db = df_update_db[list_columns_order_final]
print(f'✓ Nuestra base final cuenta con {len(df_update_db)} registros')




SECCIÓN 8: AJUSTES FINALES

🔍 Verificando columnas críticas para final adjustment:
  ✓ DEDUCTIBLE 202509: 1 valores rellenados
  ✓ LoB-Inward BDX: 966 valores rellenados
  ✓ CLAIM NUMBER ORIGINAL: 2 valores rellenados



✓ Reordenando columnas (Orden final: 59 columnas)
✓ Nuestra base final cuenta con 4257 registros


In [362]:
print(df_update_db.columns)
df_update_db.columns[df_update_db.columns.duplicated()]

print("Columnas actuales:")
print(df_update_db.columns.tolist())
print("Total columnas:", len(df_update_db.columns))


Index(['YEAR OF THE RESERVE', 'ACCIDENT YEAR', 'CLAIMS-ID', 'CLAIM NUMBER',
       'DATE OF LOSS', 'DATE OF REPORT', 'ATTRITIONAL/LARGE', 'INWARD POLICY',
       'INWARD POLICY N°', 'POLICY PERIOD START DATE',
       'POLICY PERIOD END DATE', 'SUBSIDIARY', 'StandRe LoB',
       'StandRe detailed LoB', 'LoB-Inward', 'COVER', 'COVERAGE', 'STATUS',
       'REF. PEMEX', 'LOCATION', 'DESCRIPTION', 'Cumulative SALVAGE',
       'Cumulative EXPENSES PAID', 'Cumulative VALUATION EXPENSES',
       'Cumulative CLAIMS PAID', 'Total Claims Paid Inward',
       'Total Claims Paid no Alae', 'OSLR Inward', 'Inward Incurred',
       'Inward Incurred no Alae', 'GROSS RESERVE', 'DEDUCTIBLE', 'KOT SHARE',
       'UW Year', 'Nat Cat', 'KEY LOB', 'FLAG CONTA', 'NOTAS', 'KEY DED',
       'GROSS RESERVE 202504', 'DEDUCTIBLE 202504',
       'Cumulative EXPENSES PAID 202504',
       'Cumulative VALUATION EXPENSES 202504', 'Cumulative CLAIMS PAID 202504',
       'OSLR Inward 202504', 'CLAIM NUMBER ORIGINAL BDX',

In [363]:
# =============================================================================
#  Validación de pagos antes de exportar
# =============================================================================

def validar_pagos_antes_exportacion(
    df_conta_final, df_update_db,
    key_col="KEY LOB", claim_col="CLAIMS-ID"
):
    """
    Valida que todo lo que está en df_conta_final
    exista en df_update_db, tanto por KEY LOB como por CLAIMS-ID.
    Detiene el código si hay faltantes.
    """
    errores = []

    # ── Validación 1: KEY LOB ──────────────────────────────────────────
    keys_conta = set(df_conta_final[key_col])
    keys_db    = set(df_update_db[key_col])
    faltantes_key = keys_conta - keys_db

    if faltantes_key:
        df_f = df_conta_final[
            df_conta_final[key_col].isin(faltantes_key)
        ][[key_col]].drop_duplicates()
        errores.append(
            f"[1] KEY LOB de df_conta_final no encontrados en df_update_db: {len(df_f)}\n"
            + df_f.to_string(index=False)
        )

    # ── Resultado ─────────────────────────────────────────────────────
    if errores:
        sep = "=" * 55
        detalle = ("\n" + "─" * 55 + "\n").join(errores)
        raise ValueError(
            f"\n{sep}\n"
            f"  EXPORTACIÓN DETENIDA — {len(errores)} problema(s)\n"
            f"{sep}\n"
            f"{detalle}\n"
            f"{sep}\n"
            f"  Corrige los errores antes de exportar."
        )

    print("✓ Validación OK — todos los pagos de df_conta_final están en df_update_db.")
# ── paso 6: validar ANTES de exportar ─────────────────
validar_pagos_antes_exportacion(
    df_conta_final = df_conta_final,
    df_update_db   = df_update_db,
)


✓ Validación OK — todos los pagos de df_conta_final están en df_update_db.


## 9. Final Export

In [364]:
# =============================================================================
# Section 9: Final Export.
# =============================================================================

# Final database after merge
if not os.path.exists(f'{ruta_procesados}/{AñoMes}/Final'):
    os.makedirs(f'{ruta_procesados}/{AñoMes}/Final')

df_update_db.to_excel(f'{ruta_procesados}/{AñoMes}/Final/{AñoMes}_Siniestros_Marine.xlsx', index=False)
df_update_db.to_pickle(f'{ruta_procesados}/{AñoMes}/Final/{AñoMes}_Siniestros_Marine.pkl')

# Final database after merge
if not os.path.exists(f'{ruta_bases}'):
    os.makedirs(f'{ruta_bases}')

df_update_db.to_excel(f'{ruta_bases}/{AñoMes}_Siniestros_Marine.xlsx', index=False)
df_update_db.to_pickle(f'{ruta_bases}/{AñoMes}_Siniestros_Marine.pkl')